# B2 · Imputers compared: mean, median, KNN and iterative, with known truth

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Insert random missing values into complete data, impute them with four methods and measure the error with respect to the true
values **only on the removed cells**, to establish which method reconstructs them best and why.

## Dataset

Synthetic: 800 rows and 5 standardized features, built from two latent factors so that they are correlated with each other
(`X_true`). 20% of the cells are removed at random (MCAR): `X_miss` and the boolean mask `mask` (True where the value is missing).

## Educational purpose

Compare the imputers with a **measurable truth**, impossible with real data. You will see why the mean has an error
equal to the standard deviation and how much is gained by whoever exploits the correlation between the columns.

## Task

1. Build the dictionary `imputers` with four objects: `"mean"` (`SimpleImputer(strategy="mean")`), `"median"`, `"knn"` (`KNNImputer(n_neighbors=5)`) and `"iterative"` (`IterativeImputer(random_state=0, max_iter=10)`).
2. For each one compute `X_imp = fit_transform(X_miss)` and save in `rmse[name]` the RMSE between `X_imp[mask]` and `X_true[mask]`: only on the missing cells.
3. Assign to `best` the name of the method with the lowest RMSE.
4. With `SimpleImputer(strategy="mean", add_indicator=True)` save in `shape_with_indicator` the shape of the output. Why does it have more columns than the input?
5. Write in `relative_rmse` a dictionary with the RMSE of each method divided by that of the mean: how much each one gains compared to the simplest method.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.experimental import enable_iterative_imputer          # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

rng = np.random.default_rng(0)
n = 800
X_true = rng.normal(size=(n, 2)) @ rng.normal(size=(2, 5)) + 0.5 * rng.normal(size=(n, 5))
X_true = (X_true - X_true.mean(axis=0)) / X_true.std(axis=0)                     # 5 standardized, correlated features
mask = rng.random(X_true.shape) < 0.20                                           # 20% of the cells missing, at random
X_miss = X_true.copy()
X_miss[mask] = np.nan
print("missing cells:", mask.mean().round(3), "| mean correlation between the features:",
      np.abs(np.corrcoef(X_true.T)[np.triu_indices(5, 1)]).mean().round(2))

## Your work

In [ ]:
imputers = {
    "mean": ...,
    "median": ...,
    "knn": ...,
    "iterative": ...,
}

In [ ]:
rmse = {}
for name, imp in imputers.items():
    ...
best = ...
rmse

In [ ]:
shape_with_indicator = ...
relative_rmse = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`X_imp = imp.fit_transform(X_miss)` and then `np.sqrt(np.mean((X_imp[mask] - X_true[mask]) ** 2))`; indexing with the boolean mask extracts only the removed cells.'):
    check(set(imputers) == {"mean", "median", "knn", "iterative"}, "four imputers", "you need the keys mean, median, knn, iterative")
    check(set(rmse) == set(imputers) and all(np.isfinite(v) for v in rmse.values()), "one RMSE per method", "rmse must have a key for every imputer")
    check(0.9 < rmse["mean"] < 1.1, "the RMSE of the mean is about 1: the standard deviation of a standardized feature",
          "imputing with the mean the error is the standard deviation of the feature, that is about 1")
    check(rmse["knn"] < rmse["mean"] and rmse["iterative"] < rmse["mean"], "exploiting the other columns beats the mean",
          "knn and iterative should have a lower RMSE than the mean")
    check(best == min(rmse, key=rmse.get), "best consistent with rmse", "best = min(rmse, key=rmse.get)")
    check(best == "iterative", "with linear relationships between the columns the iterative imputer wins",
          "with data built from linear factors, IterativeImputer should win")
    check(shape_with_indicator == (800, 10), "add_indicator adds one column for every column with missing values (5 + 5)",
          "expected shape (800, 10)")
    check(abs(relative_rmse["mean"] - 1.0) < 1e-9 and relative_rmse["iterative"] < 0.8, "the iterative imputer reduces the error by more than 20%",
          "relative_rmse = rmse[name] / rmse['mean']")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_imputers_compared.ipynb`. Open it after you pass the check, or after you have really tried.